<a href="https://colab.research.google.com/github/indrareddy12/AI-Engineer-Interview-Prep/blob/main/AIML_Enginner.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import re

def parse_experience(exp):
    exp = exp.lower().strip()

    if "fresher" in exp:
        return 0.0

    # Extract number (int or float)
    num = re.findall(r"\d+\.?\d*", exp)
    if not num:
        return None

    num = float(num[0])

    # Convert months → years
    if "month" in exp:
        return num / 12

    return num

# Test
data = ["2 yrs", "3.5", "4 years", "60 months", "Fresher"]

cleaned = [parse_experience(x) for x in data]
print(cleaned)



[2.0, 3.5, 4.0, 5.0, 0.0]


In [2]:
import pandas as pd

data = {
    "name": ["A", "A", "B"],
    "resume": ["r1", "r2", "r3"],
    "updated_at": ["2023-01-01", "2024-01-01", "2023-05-01"]
}

df = pd.DataFrame(data)
df["updated_at"] = pd.to_datetime(df["updated_at"])

# Keep latest
df = df.sort_values("updated_at").drop_duplicates("name", keep="last")

print(df)

  name resume updated_at
2    B     r3 2023-05-01
1    A     r2 2024-01-01


In [3]:
# ============================
# RESUME PARSING NLP PIPELINE
# ============================

# Install required packages (run once)
# pip install pdfplumber spacy
# python -m spacy download en_core_web_sm

import pdfplumber
import re
import spacy

# Load spaCy model for NLP (Named Entity Recognition)
nlp = spacy.load("en_core_web_sm")


# ----------------------------
# 1. EXTRACT TEXT FROM PDF
# ----------------------------
def extract_text_from_pdf(pdf_path):
    """
    Input: PDF file path
    Output: Raw text from the resume
    """
    text = ""

    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            # Extract text from each page
            extracted = page.extract_text()
            if extracted:  # Avoid None values
                text += extracted + "\n"

    return text


# ----------------------------
# 2. CLEAN TEXT
# ----------------------------
def clean_text(text):
    """
    Input: Raw resume text
    Output: Cleaned text
    """
    # Convert to lowercase for uniform processing
    text = text.lower()

    # Remove extra spaces and newlines
    text = re.sub(r'\s+', ' ', text)

    return text


# ----------------------------
# 3. EXTRACT SKILLS (RULE-BASED)
# ----------------------------
# Predefined skill database (can be expanded in real systems)
SKILL_DB = [
    "python", "java", "sql", "machine learning",
    "deep learning", "nlp", "tensorflow", "pytorch"
]

def extract_skills(text):
    """
    Input: Clean text
    Output: List of detected skills
    """
    found_skills = []

    for skill in SKILL_DB:
        # Check if skill exists in text
        if skill in text:
            found_skills.append(skill)

    return found_skills


# ----------------------------
# 4. EXTRACT EXPERIENCE (REGEX)
# ----------------------------
def extract_experience(text):
    """
    Extract experience like:
    '3 years', '2 yrs', '60 months'
    """
    # Regex to capture number + unit
    matches = re.findall(r'(\d+\.?\d*)\s*(years?|yrs?|months?)', text)

    experiences = []

    for num, unit in matches:
        num = float(num)

        # Convert months → years
        if "month" in unit:
            num = num / 12

        experiences.append(num)

    # Return max experience found (assumption: highest is total experience)
    return max(experiences) if experiences else 0


# ----------------------------
# 5. EXTRACT JOB TITLES (NLP NER)
# ----------------------------
def extract_entities(text):
    """
    Use spaCy to extract entities like ORG, PERSON, etc.
    (Job titles are tricky — often need custom models in real systems)
    """
    doc = nlp(text)

    entities = []
    for ent in doc.ents:
        entities.append((ent.text, ent.label_))

    return entities


# ----------------------------
# 6. FULL PIPELINE FUNCTION
# ----------------------------
def parse_resume(pdf_path):
    """
    End-to-end pipeline:
    PDF → text → clean → extract info → structured output
    """

    # Step 1: Extract raw text
    raw_text = extract_text_from_pdf(pdf_path)

    # Step 2: Clean text
    cleaned_text = clean_text(raw_text)

    # Step 3: Extract skills
    skills = extract_skills(cleaned_text)

    # Step 4: Extract experience
    experience = extract_experience(cleaned_text)

    # Step 5: Extract entities
    entities = extract_entities(cleaned_text)

    # Step 6: Return structured data
    result = {
        "skills": skills,
        "experience_years": experience,
        "entities": entities
    }

    return result


# ----------------------------
# 7. TEST PIPELINE
# ----------------------------
if __name__ == "__main__":
    pdf_path = "sample_resume.pdf"  # replace with your resume file

    output = parse_resume(pdf_path)

    print("Parsed Resume:")
    print(output)

In [ ]:
# ============================================
# 4. EMBEDDINGS + SEMANTIC SEARCH
# 5. SALARY PREDICTION MODEL
# 6. FASTAPI SERVING LAYER
# 7. RAG PIPELINE (RETRIEVAL + GENERATION)
# 8. MONITORING (LATENCY + BASIC LOGGING)
# ============================================

# Install dependencies:
# pip install sentence-transformers fastapi uvicorn scikit-learn

from sentence_transformers import SentenceTransformer
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from fastapi import FastAPI
import time


# --------------------------------------------
# 4. EMBEDDINGS + SEMANTIC SEARCH
# --------------------------------------------

# Load embedding model (converts text → vector)
embed_model = SentenceTransformer('all-MiniLM-L6-v2')

# Example resume dataset (in real system → DB)
resumes = [
    "Python developer with 3 years experience in machine learning",
    "Java backend engineer with 5 years experience",
    "Data scientist with deep learning and NLP experience"
]

# Convert resumes → embeddings (vector representation)
resume_embeddings = embed_model.encode(resumes)


def semantic_search(query):
    """
    Input: user query (e.g., 'ML engineer')
    Output: most relevant resume
    """
    # Convert query → vector
    query_embedding = embed_model.encode([query])

    # Compute similarity (dot product)
    scores = np.dot(resume_embeddings, query_embedding.T)

    # Get best matching resume
    best_index = np.argmax(scores)

    return resumes[best_index]


# --------------------------------------------
# 5. SALARY PREDICTION MODEL
# --------------------------------------------

# Example features:
# [experience_years, skill_score]
X = [
    [3, 6],
    [5, 8],
    [1, 3],
    [4, 7]
]

# Salary in LPA (example data)
y = [8, 15, 4, 12]

# Train model (tree-based → robust, interpretable)
salary_model = RandomForestRegressor()
salary_model.fit(X, y)


def predict_salary(experience, skill_score):
    """
    Input: features
    Output: predicted salary
    """
    return salary_model.predict([[experience, skill_score]])[0]


# --------------------------------------------
# 6. FASTAPI (MODEL SERVING)
# --------------------------------------------

app = FastAPI()


@app.get("/")
def home():
    return {"message": "Resume AI System Running"}


@app.post("/search_resume")
def search_resume(query: str):
    """
    API endpoint for semantic resume search
    """
    result = semantic_search(query)
    return {"best_match": result}


@app.post("/predict_salary")
def salary_api(experience: float, skill_score: float):
    """
    API endpoint for salary prediction
    """
    salary = predict_salary(experience, skill_score)
    return {"predicted_salary": salary}


# Run server:
# uvicorn main:app --reload


# --------------------------------------------
# 7. RAG PIPELINE (RETRIEVAL + GENERATION)
# --------------------------------------------

def fake_llm(prompt):
    """
    Mock LLM (replace with OpenAI / local model in real system)
    """
    return f"Generated answer based on: {prompt}"


def rag_pipeline(query):
    """
    Step 1: Retrieve relevant document
    Step 2: Pass context to LLM
    Step 3: Generate grounded response
    """
    # Retrieve relevant resume
    context = semantic_search(query)

    # Create prompt with context
    prompt = f"Context: {context}\nQuestion: {query}\nAnswer:"

    # Generate response
    response = fake_llm(prompt)

    return response


# --------------------------------------------
# 8. MONITORING (LATENCY + LOGGING)
# --------------------------------------------

def monitored_prediction(experience, skill_score):
    """
    Wrap prediction with monitoring
    """
    start_time = time.time()

    # Run prediction
    result = predict_salary(experience, skill_score)

    # Measure latency
    latency = time.time() - start_time

    # Log output (in real system → send to monitoring tool)
    print(f"Prediction: {result}, Latency: {latency:.4f} sec")

    return result


# --------------------------------------------
# TEST EVERYTHING
# --------------------------------------------
if __name__ == "__main__":

    print("---- Semantic Search ----")
    print(semantic_search("machine learning engineer"))

    print("\n---- Salary Prediction ----")
    print(predict_salary(4, 7))

    print("\n---- RAG ----")
    print(rag_pipeline("Who has ML experience?"))

    print("\n---- Monitoring ----")
    monitored_prediction(5, 8)

In [4]:
# ============================================
# DECISION TREE CLASSIFIER (FROM SCRATCH)
# ============================================

import numpy as np
from collections import Counter


# --------------------------------------------
# 1. ENTROPY FUNCTION
# --------------------------------------------
def entropy(y):
    """
    Measures how 'impure' the labels are.

    If all labels are same → entropy = 0 (pure)
    If mixed → entropy > 0

    Example:
    y = [0,0,1,1,1]
    """
    counts = Counter(y)  # Count occurrences of each class
    probs = [count / len(y) for count in counts.values()]  # Convert to probabilities

    # Apply entropy formula: -sum(p * log2(p))
    return -sum(p * np.log2(p) for p in probs if p > 0)


# --------------------------------------------
# 2. SPLIT FUNCTION
# --------------------------------------------
def split(X, y, feature_index, threshold):
    """
    Splits dataset into left and right based on condition:
    X[:, feature_index] <= threshold

    Returns:
    X_left, X_right, y_left, y_right
    """

    # Boolean masks (True/False arrays)
    left_mask = X[:, feature_index] <= threshold
    right_mask = X[:, feature_index] > threshold

    # Split data using masks
    return X[left_mask], X[right_mask], y[left_mask], y[right_mask]


# --------------------------------------------
# 3. INFORMATION GAIN
# --------------------------------------------
def information_gain(X, y, feature_index, threshold):
    """
    Measures how good a split is.

    IG = parent_entropy - weighted_child_entropy
    """

    # Entropy before split
    parent_entropy = entropy(y)

    # Split data
    X_left, X_right, y_left, y_right = split(X, y, feature_index, threshold)

    # If split is useless (one side empty), ignore
    if len(y_left) == 0 or len(y_right) == 0:
        return 0

    # Sizes
    n = len(y)
    n_left, n_right = len(y_left), len(y_right)

    # Weighted entropy after split
    child_entropy = (
        (n_left / n) * entropy(y_left) +
        (n_right / n) * entropy(y_right)
    )

    # Information Gain = improvement
    return parent_entropy - child_entropy


# --------------------------------------------
# 4. FIND BEST SPLIT
# --------------------------------------------
def best_split(X, y):
    """
    Try all features and thresholds.
    Return the one with highest information gain.
    """

    best_feature = None
    best_threshold = None
    best_gain = -1

    n_features = X.shape[1]

    # Loop over each feature (column)
    for feature in range(n_features):

        # Try all unique values as thresholds
        thresholds = np.unique(X[:, feature])

        for threshold in thresholds:
            gain = information_gain(X, y, feature, threshold)

            # Keep track of best split
            if gain > best_gain:
                best_gain = gain
                best_feature = feature
                best_threshold = threshold

    return best_feature, best_threshold


# --------------------------------------------
# 5. NODE CLASS (TREE STRUCTURE)
# --------------------------------------------
class Node:
    def __init__(self, feature=None, threshold=None, left=None, right=None, value=None):
        """
        A node can be:
        - Decision node → has feature & threshold
        - Leaf node → has value (class)
        """
        self.feature = feature
        self.threshold = threshold
        self.left = left
        self.right = right
        self.value = value  # If not None → leaf


# --------------------------------------------
# 6. DECISION TREE CLASS
# --------------------------------------------
class DecisionTree:
    def __init__(self, max_depth=5):
        self.max_depth = max_depth
        self.root = None

    # ----------------------------
    # BUILD TREE (RECURSIVE)
    # ----------------------------
    def build_tree(self, X, y, depth=0):
        """
        Recursively builds the decision tree
        """

        # STOP CONDITION 1: All labels are same → pure node
        if len(set(y)) == 1:
            return Node(value=y[0])

        # STOP CONDITION 2: Max depth reached
        if depth >= self.max_depth:
            # Return majority class
            most_common = Counter(y).most_common(1)[0][0]
            return Node(value=most_common)

        # Find best split
        feature, threshold = best_split(X, y)

        # If no valid split found
        if feature is None:
            most_common = Counter(y).most_common(1)[0][0]
            return Node(value=most_common)

        # Split data
        X_left, X_right, y_left, y_right = split(X, y, feature, threshold)

        # Recursively build left & right subtrees
        left_child = self.build_tree(X_left, y_left, depth + 1)
        right_child = self.build_tree(X_right, y_right, depth + 1)

        # Return decision node
        return Node(feature, threshold, left_child, right_child)

    # ----------------------------
    # TRAIN
    # ----------------------------
    def fit(self, X, y):
        self.root = self.build_tree(X, y)

    # ----------------------------
    # PREDICT ONE SAMPLE
    # ----------------------------
    def predict_one(self, x, node):
        """
        Traverse tree until leaf node
        """

        # If leaf → return class
        if node.value is not None:
            return node.value

        # Decide direction based on condition
        if x[node.feature] <= node.threshold:
            return self.predict_one(x, node.left)
        else:
            return self.predict_one(x, node.right)

    # ----------------------------
    # PREDICT MULTIPLE
    # ----------------------------
    def predict(self, X):
        return np.array([self.predict_one(x, self.root) for x in X])


# --------------------------------------------
# 7. TEST THE TREE
# --------------------------------------------
if __name__ == "__main__":

    # Example dataset
    # [experience, skill_score]
    X = np.array([
        [1, 2],
        [2, 3],
        [3, 6],
        [4, 7],
        [5, 8]
    ])

    # Labels (0 = reject, 1 = hire)
    y = np.array([0, 0, 1, 1, 1])

    # Create tree
    tree = DecisionTree(max_depth=3)

    # Train
    tree.fit(X, y)

    # Predict
    predictions = tree.predict(X)

    print("Predictions:", predictions)

Predictions: [0 0 1 1 1]


In [6]:
import numpy as np

def PCA(X, n_components):
    """
    Perform PCA on dataset X

    Parameters:
    X : numpy array (n_samples, n_features)
    n_components : number of dimensions to reduce to

    Returns:
    X_reduced : transformed data
    components : principal components (directions)
    """

    # ----------------------------
    # 1. MEAN CENTERING
    # ----------------------------
    # Subtract mean so data is centered around 0
    mean = np.mean(X, axis=0)
    X_centered = X - mean

    # ----------------------------
    # 2. COVARIANCE MATRIX
    # ----------------------------
    # Measures how features vary together
    cov_matrix = np.cov(X_centered, rowvar=False)

    # ----------------------------
    # 3. EIGEN DECOMPOSITION
    # ----------------------------
    # Eigenvectors → directions
    # Eigenvalues → importance (variance)
    eigenvalues, eigenvectors = np.linalg.eig(cov_matrix)

    # ----------------------------
    # 4. SORT EIGENVECTORS
    # ----------------------------
    # Sort by eigenvalues (descending)
    sorted_indices = np.argsort(eigenvalues)[::-1]

    eigenvalues = eigenvalues[sorted_indices]
    eigenvectors = eigenvectors[:, sorted_indices]

    # ----------------------------
    # 5. SELECT TOP COMPONENTS
    # ----------------------------
    components = eigenvectors[:, :n_components]

    # ----------------------------
    # 6. PROJECT DATA
    # ----------------------------
    # Convert data into new lower-dimensional space
    X_reduced = np.dot(X_centered, components)

    return X_reduced, components


# ----------------------------
# TEST EXAMPLE
# ----------------------------
if __name__ == "__main__":

    # Example dataset (2D → reduce to 1D)
    X = np.array([
        [2, 3],
        [3, 4],
        [4, 5],
        [5, 6]
    ])

    X_reduced, components = PCA(X, n_components=1)

    print("Reduced Data:\n", X_reduced)
    print("\nPrincipal Component:\n", components)



    #Eigenvalues:
#[largest → smallest]

#Eigenvectors:
#[best direction first]

Reduced Data:
 [[-2.12132034]
 [-0.70710678]
 [ 0.70710678]
 [ 2.12132034]]

Principal Component:
 [[0.70710678]
 [0.70710678]]


In [7]:
#k measn form sratch
# 1. decide clsuters
# 2. select random centroids
#3. assign centroids
#4. move sentorids
# 5. check finish

import random
import numpy as np

class KMeans:
    def __init__(self,n_clusters=2,max_iter=100):
        self.n_clusters = n_clusters
        self.max_iter = max_iter
        self.centroids = None

    def fit_predict(self,X):

        random_index = random.sample(range(0,X.shape[0]),self.n_clusters)
        self.centroids = X[random_index]

        for i in range(self.max_iter):
            # assign clusters
            cluster_group = self.assign_clusters(X)
            old_centroids = self.centroids
            # move centroids
            self.centroids = self.move_centroids(X,cluster_group)
            # check finish
            if (old_centroids == self.centroids).all():
                break

        return cluster_group

    def assign_clusters(self,X):
        cluster_group = []
        distances = []

        for row in X:
            for centroid in self.centroids:
                distances.append(np.sqrt(np.dot(row-centroid,row-centroid)))
            min_distance = min(distances)
            index_pos = distances.index(min_distance)
            cluster_group.append(index_pos)
            distances.clear()

        return np.array(cluster_group)

    def move_centroids(self,X,cluster_group):
        new_centroids = []

        cluster_type = np.unique(cluster_group)

        for type in cluster_type:
            new_centroids.append(X[cluster_group == type].mean(axis=0))

        return np.array(new_centroids)
